# The JDM lab as its own study (v2)
### Load the JDM lab's files → work out who their samples are → check against our numbers → (display-only) merge

**What this notebook does.**

1. Reads **our** cohort from notebook 2 (it no longer repeats notebook 2's code).
2. Loads the **JDM lab's** processed files and recovers who each sample is.
3. Builds the JDM lab as **its own study**: a sample table and an expression matrix, saved for notebook 4.
4. Checks that our RPKM reproduces theirs on the few samples both datasets contain.
5. Builds a **display-only** merged matrix for plots, and nothing more.

<div class="alert alert-warning">
<b>What changed from v1, and why.</b>
<ul>
<li><b>No more joint batch-corrected matrix for inference.</b> v1 merged both datasets, quantile-normalised jointly, then subtracted a dataset term. The NIH-NLM notebooks (46-49b) show why this cannot work when each lab holds one kind of sample: lab and disease are inseparable, and the RUVg alternative removed real disease signal. In v1 the correction could only separate the JDM lab from "ours"; it could not separate our HC workbook from our Goldbach patients, because that difference is absorbed into the group terms. The old "healthy controls overlap after correction" checkpoint also could not fail, because the HC means are aligned by construction.</li>
<li><b>The JDM lab is analysed within itself.</b> JDM vs HC inside the JDM lab is the only contrast here that is free of batch. It is tested in notebook 4.</li>
<li><b>Cross-study comparison is done at the level of module scores and effect sizes</b> (notebook 4), as in their notebooks 15, 26 and 30, not by pooling expression.</li>
<li>The merged matrix is still produced, but its files are named <code>DISPLAY_ONLY_*</code> and it carries no correction.</li>
</ul>
</div>

**Roadmap**

| Part | Step | Output |
|---|---|---|
| 1 | Configuration & helpers | paths, rules, `qnorm`, `shape` |
| 2 | Patient metadata | `md`, lookups |
| 3 | Our cohort (from notebook 2) | `ours`, `counts`, `L` |
| 4 | JDM lab files and key sheet | `rp`, `qn`, `key` |
| 5 | The JDM lab as its own study | `jdm_lab_samples.csv`, `jdm_lab_expression_log2QN.csv` |
| 6 | Cross-check against our numbers | `pairs`, `cors` |
| 7 | Display-only merge | `DISPLAY_ONLY_*` |
| 8 | Processing log | `PROCESSING_LOG_notebook3.md` |

<div class="alert alert-warning">
<b>Privacy.</b> Birth date is read only to compute age, then dropped. Printed output never shows real IDs or birth dates; IDs are masked to their format (<code>JDM_181 → AAA_999</code>). All outputs are CSV.
</div>

---
# Part 1 · Configuration & helpers

🎯 **Goal:** every path and rule in one place, plus the small tools used throughout.

💡 **Why:** `LIB_CONVENTION` decides which library size our RPKM uses when compared with theirs (see Part 6). `DROP_PEOPLE_ALREADY_IN_OURS` only affects the display merge. `shape` masks an ID to its format so matching problems can be diagnosed without printing real IDs.

In [ ]:
import os, re, datetime, platform, warnings
import numpy as np, pandas as pd
warnings.filterwarnings("ignore")

MNT = "/mnt/file-systems/"
RESULTS = "/home/jovyan/session_data/endotype_results"               # the ONE results folder
V2  = RESULTS + "/rnaseq_out_v2/"                       # written by notebook 2
METADATA_PATH = MNT + "MetadataPatients_NoPHI.xlsx"
JDM_RPKM = MNT + "4b_rpkm.csv"
JDM_QN   = MNT + "4a_log2QN.csv"
JDM_KEY  = MNT + "jdm-HC RNASEQ sample ID.csv"

OURS_SAMPLES    = V2 + "rnaseq_samples.csv"                   # one per patient (notebook 2)
OURS_COUNTS_ALL = V2 + "rnaseq_counts_all_samples.csv.gz"     # every sample with counts (notebook 2)
LENGTHS         = V2 + "gene_lengths.csv"                     # cached by notebook 2
OUT_DIR = V2                                                   # outputs sit next to notebook 2's

LIB_CONVENTION = "their_gene_set"        # RPKM library size for the comparison: "their_gene_set" or "all_genes"
DROP_PEOPLE_ALREADY_IN_OURS = True       # display merge only: skip JDM-lab samples from people already in our kept set
MAKE_DISPLAY_MERGE = True                # Part 7 (plots only)

for p in (OURS_SAMPLES, OURS_COUNTS_ALL, LENGTHS, METADATA_PATH, JDM_RPKM, JDM_QN, JDM_KEY):
    assert os.path.exists(p), f"missing input: {p}  (run notebook 2 first if it is one of its outputs)"
assert os.access(OUT_DIR, os.W_OK), f"{OUT_DIR} is not writable"

def normalize_id(x):
    s = "" if x is None else str(x).strip()
    if s.lower() in ("nan", "none", "nat", ""):
        return ""
    s = re.sub(r"\.0$", "", s)
    s = s.upper().replace("-", "").replace(" ", "").replace("_", "")
    if s.startswith("CHLD"):                 # CHLD0047 (RNA files) == LD0047 (metadata)
        s = s[2:]
    return s

def zs(s):
    """drop zero-padding that follows a letter: JDM0181 -> JDM181"""
    return re.sub(r"(?<=[A-Z])0+(?=\d)", "", s)

def shape(x):
    """mask an ID so only its format shows: 'JDM_181' -> 'AAA_999'"""
    return re.sub(r"\d", "9", re.sub(r"[A-Za-z]", "A", str(x)))

def norm_colname(c):
    return re.sub(r"[^a-z0-9]", "", str(c).lower())

DATE_MIN, DATE_MAX = pd.Timestamp("2008-01-01"), pd.Timestamp.today().normalize()

def qnorm(df):
    """Quantile normalisation across the columns of df (no NaN allowed)."""
    mean_sorted = np.sort(df.values, axis=0).mean(axis=1)
    ranks = df.rank(method="first").astype(int).values - 1
    return pd.DataFrame(mean_sorted[ranks], index=df.index, columns=df.columns)

def finfo(p):
    return (f"{os.path.basename(p)}  ({os.path.getsize(p)/1e6:.1f} MB, modified "
            f"{datetime.datetime.fromtimestamp(os.path.getmtime(p)):%Y-%m-%d})")

---
# Part 2 · Patient metadata

🎯 **Goal:** the master patient sheet and fast lookups, used to recover who the JDM-lab samples are.

💡 **Why:** besides the ID lookup, this builds a **zero-padding-insensitive** lookup, a **unique-initials** lookup (shared initials are never auto-resolved: a wrong patient is worse than a blank one), and the IDs hidden in the **Notes** column (a weak last resort).

In [ ]:
sheets = pd.read_excel(METADATA_PATH, sheet_name=None, dtype=str)
md = max(sheets.values(), key=lambda d: sum(bool(re.search(r"enroll|diagnos|initial", str(c), re.I))
                                            for c in d.columns))
md.columns = [str(c).strip() for c in md.columns]
md = md.dropna(how="all").reset_index(drop=True)

def fc(*names):
    nm = {norm_colname(c): c for c in md.columns}
    return next((nm[norm_colname(n)] for n in names if norm_colname(n) in nm), None)

COL = {"enroll": fc("Enrollment ID"), "alt": fc("Alternate Enrollment ID"),
       "initials": fc("Initials"), "dob": fc("Birth Date", "DOB"), "sex": fc("Sex", "Gender"),
       "race": fc("Race"), "ethnicity": fc("Ethnicity"), "status": fc("Study Status"),
       "dx1": fc("Diagnosis1"), "dx2": fc("Diagnosis2"), "note": fc("Note", "Notes")}
print("metadata:", len(md), "rows; columns not found:", [k for k, v in COL.items() if v is None])

meta_lookup, zs_lookup, ini_rows, notes_norm = {}, {}, {}, {}
for i, r in md.iterrows():
    for kc in (COL["enroll"], COL["alt"]):
        k = normalize_id(r.get(kc)) if kc else ""
        if k:
            meta_lookup.setdefault(k, i)
            zs_lookup.setdefault(zs(k), i)
    ini = str(r.get(COL["initials"]) or "").strip().upper() if COL["initials"] else ""
    if ini:
        ini_rows.setdefault(ini, set()).add(i)
    if COL["note"]:
        notes_norm[i] = normalize_id(r.get(COL["note"]))
INI_UNIQUE_ROW = {k: next(iter(v)) for k, v in ini_rows.items() if len(v) == 1}

def meta_value(i, key):
    c = COL.get(key)
    if i < 0 or not c:
        return ""
    v = md.at[i, c]
    return "" if pd.isna(v) else str(v).strip()

HEALTHY = re.compile(r"^hc$|health|control", re.I)
MYOSITIS = re.compile(r"myosit|dermatomyos|jdm", re.I)

---
# Part 3 · Our cohort, from notebook 2

🎯 **Goal:** load what notebook 2 produced, instead of repeating its logic.

💡 **Why:** v1 of this notebook contained a copy of notebook 2's code, and the two copies drifted apart (different `DX_MAP`, different empty-sample handling), producing different cohorts. Reading the saved outputs gives **one source of truth**. Here we need the one-per-patient sample sheet (to avoid counting a person twice in the display merge), the counts of every sample (for the shared-sample check), and the cached gene lengths.

In [ ]:
ours = pd.read_csv(OURS_SAMPLES, dtype=str, keep_default_na=False)
ours["age_yrs"] = pd.to_numeric(ours["age_yrs"], errors="coerce")
print("our one-per-patient cohort:", len(ours), "samples")
print(ours["dx_clean"].value_counts().to_string())

counts_all = pd.read_csv(OURS_COUNTS_ALL, index_col=0)
gene_names = counts_all["Gene_Name"]
counts = counts_all.drop(columns="Gene_Name")
print(f"\ncounts of every sample with counts: {counts.shape[0]} genes x {counts.shape[1]} samples")

lengths = pd.read_csv(LENGTHS, index_col=0).iloc[:, 0]
versioned = counts.index.isin(lengths.index).mean() > 0.5
lkp, gkey = lengths.copy(), counts.index
if not versioned:
    lkp.index = lkp.index.str.split(".").str[0]
    lkp = lkp[~lkp.index.duplicated()]
    gkey = counts.index.str.split(".").str[0]
L = pd.Series(lkp.reindex(gkey).values / 1000.0, index=counts.index)
ok = L.notna() & (L > 0)
print(f"gene length found for {L.notna().mean():.1%} of {len(L)} genes")

---
# Part 4 · The JDM lab's files and key sheet

🎯 **Goal:** load the JDM lab's processed expression files and find out *who* each sample is.

💡 **Why this is its own problem:** the lab shipped matrices keyed by their own sample IDs. Demographics come through a separate **key sheet** linking their IDs to NIH enrollment IDs and initials, which in turn link to our metadata. Each link can fail, so each is tried in order of reliability.

### 4.1 Load their expression matrices

💡 `load_lab` splits the few header rows (RPKM: Cohort, ID; QN: Cohort, MSA, ID) from the gene rows and transposes them so each sample is a row, then labels expression columns with the sample **ID**. `4b_rpkm.csv` is their RPKM; `4a_log2QN.csv` is their log2, quantile-normalised version, which becomes the JDM-lab study's analysis matrix.

In [ ]:
def load_lab(path, n_meta):
    raw = pd.read_csv(path, low_memory=False)
    g = raw.columns[0]
    meta = raw.iloc[:n_meta].set_index(g).T
    expr = raw.iloc[n_meta:].set_index(g).apply(pd.to_numeric, errors="coerce")
    return meta, expr

m_b, rp = load_lab(JDM_RPKM, 2)                  # rows: Cohort, ID
m_a, qn = load_lab(JDM_QN, 3)                    # rows: Cohort, MSA, ID
m_b.index = m_b["ID"].values
rp.columns = m_b["ID"].values
m_a.index = m_a["ID"].values
qn.columns = m_a["ID"].values
print(f"JDM lab RPKM: {rp.shape}; QN: {qn.shape}; duplicated symbols in RPKM: {int(rp.index.duplicated().sum())}")
print(m_b["Cohort"].value_counts().to_string())
print("same sample IDs in both files:", set(rp.columns) == set(qn.columns))

### 4.2 The key sheet and its dates

💡 The key sheet mixes **Excel serial numbers** (`40413`) with `m/d/yy` text, and two-digit years need a pivot (≤30 means 2000s). Dates before 2000 or after today are rejected. R appended `...2`-style suffixes to duplicate headers, so `grp` strips them to recover the real group.

In [ ]:
KEY_DATE_MIN = pd.Timestamp("2000-01-01")

def parse_key_date(v):
    """Excel serial numbers (40413) and m/d/yy text are both handled."""
    v = str(v).strip()
    if not v:
        return pd.NaT
    if re.fullmatch(r"\d{5}(\.\d+)?", v):
        d = pd.Timestamp("1899-12-30") + pd.Timedelta(days=float(v))
    else:
        m = re.search(r"(\d{1,2})/(\d{1,2})/(\d{2,4})", v)
        if not m:
            return pd.NaT
        mo, da, yr = (int(x) for x in m.groups())
        if yr < 100:
            yr += 2000 if yr <= 30 else 1900
        try:
            d = pd.Timestamp(year=yr, month=mo, day=da)
        except ValueError:
            return pd.NaT
    return d if KEY_DATE_MIN <= d <= DATE_MAX else pd.NaT

key = pd.read_csv(JDM_KEY, dtype=str, keep_default_na=False)
key.columns = [c.strip() for c in key.columns]
key["grp"] = key["Diagnosis"].str.replace(r"\.\.\.\d+$", "", regex=True)
key["_date"] = key["Date"].map(parse_key_date)
print(f"key sheet: {len(key)} rows; dates parsed: {int(key['_date'].notna().sum())}")

### 4.3 Match key-sheet rows to our patient metadata

💡 **Why a ladder:** no single field is reliable. Steps run from most to least trustworthy and the method is recorded in `matched_by`:

| Step | Method | Trust |
|---|---|---|
| 1 | NIH Enroll ID (plain, or with `R` prefix) | high |
| 2 | Initials interpreted as an ID | high |
| 3 | ID ignoring zero-padding (`JDM0181` = `JDM181`) | high |
| 4 | **Unique** initials, only if the diagnosis does not conflict | medium |
| 5 | The ID inside the metadata **Notes**, only if exactly one row contains it and the token has 6+ characters | weak |

Age is `(key-sheet date − birth date) / 365.25`; the birth date is then discarded. Unmatched rows are reported by **format only** (`shape`).

In [ ]:
def match_key(r):
    """NIH Enroll ID (plain / R prefix) -> initials as an ID -> ID ignoring zero-padding ->
    unique initials (diagnosis must not conflict) -> the ID inside the metadata Notes (weak)."""
    k = normalize_id(r["NIH Enroll ID"])
    ini = str(r["Initials"]).strip()
    ini_up, ini_id = ini.upper(), normalize_id(ini)
    if k and k in meta_lookup:
        return meta_lookup[k], "nih_enroll_id"
    if k.isdigit() and ("R" + k) in meta_lookup:
        return meta_lookup["R" + k], "nih_enroll_id (R prefix)"
    if ini_id and ini_id in meta_lookup:
        return meta_lookup[ini_id], "initials_as_id"
    for tok in (ini_id, k):
        if tok and not tok.isdigit() and zs(tok) in zs_lookup:
            return zs_lookup[zs(tok)], "id_zero_padding"
    i = INI_UNIQUE_ROW.get(ini_up, -1)
    if i >= 0:
        dx = meta_value(i, "dx1")
        ok_dx = (dx == "" or (r["grp"] == "HC" and HEALTHY.search(dx))
                 or (r["grp"] == "JDM" and MYOSITIS.search(dx)))
        return (i, "initials_unique") if ok_dx else (-1, "initials_diagnosis_conflict")
    for tok in (ini_id, k):
        if len(tok) >= 6:
            hits = [j for j, n in notes_norm.items() if n and tok in n]
            if len(hits) == 1:
                return hits[0], "found_in_notes"
    return -1, "none"

mk = key.apply(lambda r: pd.Series(match_key(r), index=["_row", "matched_by"]), axis=1)
key = pd.concat([key, mk], axis=1)
print("key sheet matched to metadata by:")
print(pd.crosstab(key["grp"], key["matched_by"]).to_string())

kdob = pd.to_datetime(key["_row"].map(lambda i: meta_value(i, "dob")), errors="coerce")
key["age_yrs"] = ((key["_date"] - kdob).dt.days / 365.25).round(1)
key.loc[(key["age_yrs"] < 0) | (key["age_yrs"] > 120), "age_yrs"] = np.nan
for col in ("sex", "race", "ethnicity"):
    key[col] = key["_row"].map(lambda i, col=col: meta_value(i, col))
key["patient_key"] = [normalize_id(meta_value(i, "enroll")) or normalize_id(kid)
                      for i, kid in zip(key["_row"], key["NIH Enroll ID"])]
print("\nage known (key rows):",
      key.groupby("grp")["age_yrs"].apply(lambda s: f"{int(s.notna().sum())}/{len(s)}").to_dict())
un = key[key["_row"] < 0]
print(f"still unmatched: {len(un)}. Formats (masked, no real IDs shown):")
print(pd.DataFrame({"grp": un["grp"], "Initials_format": un["Initials"].map(shape),
                    "NIH_Enroll_ID_format": un["NIH Enroll ID"].map(shape)}
                   ).value_counts().head(12).to_string())
key = key.drop_duplicates("ID").set_index("ID")

### 👀 Checkpoint 4: how were the JDM-lab samples identified?

Strong methods (`nih_enroll_id*`, `initials_as_id`, `id_zero_padding`) count as `demographics_trusted`. Everything else carries age and sex from a weaker link, or none. **Many unmatched rows means the patient sheet does not contain those people** (many key-sheet "NIH Enroll ID" values look like short row numbers, not enrollment IDs). Better matching will not fix that; age and sex would have to come from another file, such as the lab's own clinical sheet (`JDM IFN scores OADR.xlsx` is a candidate that has not been checked).

In [ ]:
import matplotlib.pyplot as plt
ct = pd.crosstab(key["grp"], key["matched_by"])
ax = ct.plot.barh(stacked=True, figsize=(11, 3.4))
ax.invert_yaxis(); ax.set_title("JDM-lab key-sheet rows by match method"); ax.set_ylabel("")
ax.legend(bbox_to_anchor=(1.01, 1), loc="upper left", fontsize=8)
plt.tight_layout(); plt.show()

---
# Part 5 · The JDM lab as its own study

🎯 **Goal:** a self-contained study: a **sample table** and an **expression matrix**, saved for notebook 4. Nothing here is merged with our data.

💡 **Why this is the right unit:** the contrast JDM vs HC **inside** the JDM lab is the only one in this project that a batch effect cannot explain, because both groups were processed together. Their notebook 40/46 show the alternative (comparing patients with healthy controls from another lab) gives results where about three quarters of genes differ and lab and disease cannot be told apart.

- **Expression matrix:** their `4a_log2QN.csv` (log2 RPKM, quantile-normalised **within** their 116 samples). Part 6 verifies that this is exactly `log2(RPKM+1)` followed by quantile normalisation.
- **Sample table:** group (HC or JDM), MSA, PGA where present, demographics from the key sheet with the match method, and two flags: `shared_with_ours` (the same sample also appears in our RNA files) and `patient_repeated_within_jdm_lab` (the same person has more than one sample; with so many unmatched IDs this flag can only catch some repeats, so tests treat the samples as independent only approximately).

In [ ]:
# which JDM-lab samples are also in our RNA files? (sample names end in "-<their ID>")
pairs = {j: s for j in rp.columns for s in counts.columns if s.endswith("-" + j)}
print(f"samples present in both datasets: {len(pairs)} of {rp.shape[1]}")

jm = pd.DataFrame(index=rp.columns)
jm["sample_id"] = jm.index
jm["dataset"] = "jdm_lab"
jm["group"] = m_b["Cohort"].reindex(jm.index).values
jm["msa"] = m_a["MSA"].reindex(jm.index).values if "MSA" in m_a.columns else ""
jm["pga"] = key["PGA"].reindex(jm.index).values if "PGA" in key.columns else ""
for col in ("sex", "race", "ethnicity", "patient_key", "age_yrs"):
    jm[col] = key[col].reindex(jm.index).replace("", np.nan).values
jm["demographics_match"] = key["matched_by"].reindex(jm.index).fillna("not in key sheet").values
jm["demographics_trusted"] = (jm["demographics_match"].str.startswith("nih_enroll_id")
                              | jm["demographics_match"].isin(["initials_as_id", "id_zero_padding"]))
jm["age_date_source"] = np.where(key["_date"].reindex(jm.index).notna().values, "key_sheet_date", "none")
jm["patient_key"] = jm["patient_key"].fillna("")
jm["patient_repeated_within_jdm_lab"] = jm["patient_key"].ne("") & jm["patient_key"].duplicated(keep=False)
jm["shared_with_ours"] = jm.index.isin(pairs.keys())
jm["source_file"] = os.path.basename(JDM_RPKM)

jm.to_csv(os.path.join(OUT_DIR, "jdm_lab_samples.csv"), index=False)
qn.to_csv(os.path.join(OUT_DIR, "jdm_lab_expression_log2QN.csv"))
print("\nJDM-lab study written:", os.path.join(OUT_DIR, "jdm_lab_samples.csv"), "and jdm_lab_expression_log2QN.csv")
print("\nsamples by group:"); print(jm["group"].value_counts().to_string())
print("\nage known / trusted demographics by group:")
print(jm.groupby("group").agg(n=("sample_id", "size"), age_known=("age_yrs", lambda s: int(s.notna().sum())),
                              trusted=("demographics_trusted", "sum"),
                              repeated_patient=("patient_repeated_within_jdm_lab", "sum"),
                              shared_with_ours=("shared_with_ours", "sum")).to_string())

---
# Part 6 · Does our RPKM reproduce theirs?

🎯 **Goal:** before any cross-dataset use, check that the two pipelines agree on samples that appear in **both**.

💡 **Why it matters:** if our pipeline and theirs disagree on the *same biological sample*, every cross-dataset difference could be an artifact. Two tests:

- **Test 1: our RPKM vs their RPKM, same samples**, under two library-size conventions: `all_genes` (reads over all genes) and `their_gene_set` (reads only inside their gene set, since their pipeline likely counted only those). Reported: **Spearman** (do gene rankings agree?) and the **median ratio** on well-expressed genes (do the *levels* agree?).
- **Test 2: rebuild their QN file from their RPKM file.** If `log2(RPKM+1)` then quantile normalisation reproduces `4a_log2QN.csv`, we know their recipe, and the matrix saved in Part 5 is exactly that recipe.

**How to read the result.** In the previous run, Spearman was 0.81 to 0.86 and the median ratio 0.84 to 0.88 with `their_gene_set`, against 0.57 to 0.64 with all genes. That is close but not identical, which is one more reason not to treat the two datasets as interchangeable.

In [ ]:
sym_all = pd.Series(gene_names.reindex(counts.index).values, index=counts.index)
in_theirs = sym_all.isin(rp.index)
lib_theirs = counts[in_theirs].sum()

def rpkm_by_symbol(libsize, sample_ids):
    r = counts.loc[ok, sample_ids].div(L[ok], axis=0).div(libsize[sample_ids] / 1e6, axis=1)
    best = r.mean(axis=1).groupby(sym_all[ok].values).idxmax()     # best Ensembl row per symbol
    out = r.loc[best.values]
    out.index = best.index
    return out

if pairs:
    sids = sorted(set(pairs.values()))
    rows = []
    for conv, libsize_ in (("all_genes", counts.sum()), ("their_gene_set", lib_theirs)):
        rr = rpkm_by_symbol(libsize_, sids)
        sg = rr.index.intersection(rp.index)
        for j, s in pairs.items():
            x, y = rr.loc[sg, s], rp.loc[sg, j]
            hi = y > 1
            rows.append((conv, s[:34], round(x.rank().corr(y.rank()), 3),
                         round(float((x[hi] / y[hi]).median()), 2)))
    print(pd.DataFrame(rows, columns=["library_size", "our_sample", "spearman",
                                      "median_ratio_ours_over_theirs"]).to_string(index=False))
    print("share of our reads inside their gene set (median over samples):",
          round(float((lib_theirs / counts.sum()).median()), 3))
else:
    print("no samples in both datasets; skipping Test 1")

common_g = rp.index.intersection(qn.index)
common_s = [c for c in rp.columns if c in qn.columns]
mine = qnorm(np.log2(rp.loc[common_g, common_s] + 1))
cors = [mine[c].corr(qn.loc[common_g, c]) for c in common_s]
print(f"\nrebuild of their 4a from their 4b via log2(x+1)+QN: median per-sample r = {np.median(cors):.4f}, "
      f"max abs diff = {np.nanmax(np.abs(mine.values - qn.loc[common_g, common_s].values)):.4f}")

### 👀 Checkpoint 6: the comparison, visually

**Left:** our RPKM vs theirs for one shared sample (log-log); points near the diagonal mean agreement. **Right:** per-sample correlation of our rebuild of their QN file; every value should be about 1.0.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
if pairs:
    j0, s0 = next(iter(pairs.items()))
    rr = rpkm_by_symbol(lib_theirs, [s0]); sg = rr.index.intersection(rp.index)
    ax[0].scatter(np.log2(rp.loc[sg, j0] + 1), np.log2(rr.loc[sg, s0] + 1), s=3, alpha=.3)
    lim = [0, ax[0].get_xlim()[1]]; ax[0].plot(lim, lim, "r-", lw=.8)
    ax[0].set_xlabel("their log2(RPKM+1)"); ax[0].set_ylabel("our log2(RPKM+1)")
    ax[0].set_title("One shared sample (their_gene_set library)")
else:
    ax[0].text(.5, .5, "no shared samples", ha="center"); ax[0].axis("off")
ax[1].hist(cors, bins=20, color="#59A14F"); ax[1].set_title("QN rebuild: per-sample r vs their 4a")
plt.tight_layout(); plt.show()

---
# Part 7 · Display-only merge

🎯 **Goal:** one matrix with both datasets, **for plots only**.

💡 **Why it is still useful, and why it is not an analysis input:** a PCA of the merged data coloured by dataset shows the size of the lab effect at a glance (in the previous run PC1, 52.5% of the variance, separated the two datasets completely). It must not be used to compare diseases: the dataset difference and the disease difference cannot be separated, and **no correction is applied here.**

- Genes are matched on **symbol**; where several Ensembl rows share one symbol, the most-expressed row is kept.
- Our RPKM uses `LIB_CONVENTION` so it is built like theirs.
- JDM-lab samples are skipped if the **same sample** is in our files, and (with `DROP_PEOPLE_ALREADY_IN_OURS`) if the **same person** is in our kept set.
- log2(RPKM+1) then quantile normalisation, jointly; only genes with a value in every sample are kept.

Files are named `DISPLAY_ONLY_*` so they cannot be mistaken for analysis inputs.

In [ ]:
if not MAKE_DISPLAY_MERGE:
    print("MAKE_DISPLAY_MERGE is False; skipping")
else:
    kept_ids = list(ours["sample_id"])
    libsize = lib_theirs if LIB_CONVENTION == "their_gene_set" else counts.sum()
    ours_r = rpkm_by_symbol(libsize, kept_ids)

    ours_people = set(ours["patient_key"])
    why = pd.Series("", index=jm.index)
    why[jm["shared_with_ours"].values] = "same sample is in our RNA files"
    if DROP_PEOPLE_ALREADY_IN_OURS:
        same_person = jm["patient_key"].isin(ours_people) & jm["patient_key"].ne("") & (why == "")
        why[same_person] = "person already in our kept set"
    jm_keep = jm[why == ""]
    print(f"JDM-lab samples: {len(jm)} -> in display merge {len(jm_keep)}; skipped: {why[why != ''].value_counts().to_dict()}")

    genes = ours_r.index.intersection(rp.index)
    X = pd.concat([ours_r.loc[genes, kept_ids], rp.loc[genes, list(jm_keep.index)]], axis=1)
    X = X.replace([np.inf, -np.inf], np.nan)
    dead = X.columns[X.isna().all(axis=0)]
    if len(dead):
        print("WARNING: samples with no usable values, dropped:", list(dead)); X = X.drop(columns=list(dead))
    n_before = X.shape[0]
    X = X.dropna(axis=0)
    print(f"genes with a value in every sample: {X.shape[0]} of {n_before}")
    QN = qnorm(np.log2(X + 1))

    top = QN.var(axis=1).sort_values(ascending=False).index[:2000]
    Z = QN.loc[top].T.values
    Z = Z - Z.mean(axis=0)
    U, S, _ = np.linalg.svd(Z, full_matrices=False)
    pcs = U[:, :3] * S[:3]
    expl = (S ** 2 / (S ** 2).sum())[:3]

    disp = pd.concat([
        ours.assign(dataset="ours", group=ours["dx_clean"])[["sample_id", "dataset", "group", "batch"]],
        jm_keep.assign(batch="jdm_lab")[["sample_id", "dataset", "group", "batch"]]], ignore_index=True)
    disp = disp.set_index("sample_id").loc[list(QN.columns)].reset_index()
    for i in range(3):
        disp[f"PC{i+1}"] = pcs[:, i]
    QN.to_csv(os.path.join(OUT_DIR, "DISPLAY_ONLY_merged_log2_RPKM_QN.csv"))
    disp.to_csv(os.path.join(OUT_DIR, "DISPLAY_ONLY_merged_samples.csv"), index=False)
    print("PCA variance explained:", [f"{e:.1%}" for e in expl])
    print(disp.groupby(["dataset", "group"])[["PC1", "PC2"]].mean().round(2).to_string())

### 👀 Checkpoint 7: how big is the lab effect?

**Left:** by dataset; **right:** by group. If the two datasets form separate clouds, that is the lab effect, and it is the reason the JDM lab is analysed on its own.

In [ ]:
if MAKE_DISPLAY_MERGE:
    fig, ax = plt.subplots(1, 2, figsize=(13, 4.8))
    for d, c in (("ours", "#4C78A8"), ("jdm_lab", "#E15759")):
        s = disp[disp["dataset"] == d]
        ax[0].scatter(s["PC1"], s["PC2"], s=14, alpha=.7, c=c, label=d)
    ax[0].legend(); ax[0].set_title("Display only, no correction: by dataset")
    for g, s in disp.groupby("group"):
        ax[1].scatter(s["PC1"], s["PC2"], s=14, alpha=.7, label=g)
    ax[1].legend(fontsize=7, ncol=2); ax[1].set_title("Display only, no correction: by group")
    for a in ax: a.set_xlabel(f"PC1 ({expl[0]:.0%})"); a.set_ylabel(f"PC2 ({expl[1]:.0%})")
    plt.tight_layout(); plt.show()

---
# Part 8 · Processing log

🎯 **Goal:** a Markdown record of what this notebook did, with the numbers filled in **from the run itself**, so it cannot go stale.

In [ ]:
Lg = ["# RNA-seq processing log (notebook 3, v2)",
      f"Generated {datetime.datetime.now():%Y-%m-%d %H:%M}. Python {platform.python_version()}, "
      f"pandas {pd.__version__}, numpy {np.__version__}.\n", "## Inputs"]
Lg += [f"- {finfo(p)}" for p in (OURS_SAMPLES, OURS_COUNTS_ALL, LENGTHS, METADATA_PATH, JDM_RPKM, JDM_QN, JDM_KEY)]
Lg += ["", "## The JDM lab as its own study",
       f"- JDM-lab samples: {len(jm)}; groups: {jm['group'].value_counts().to_dict()}.",
       "- Expression matrix: their 4a (log2 RPKM, quantile-normalised within the JDM lab).",
       f"- Rebuild of 4a from 4b via log2(RPKM+1)+QN: median per-sample r = {np.median(cors):.4f}.",
       f"- Demographics matched by: {key['matched_by'].value_counts().to_dict()}.",
       f"- Age known for {int(jm['age_yrs'].notna().sum())} of {len(jm)} samples (key-sheet date; its meaning is unconfirmed).",
       f"- Samples also present in our RNA files: {len(pairs)}.",
       "- No treatment or sample-source labels exist for these samples; all are stated to be whole blood.",
       "", "## What is NOT done, and why",
       "- No joint batch correction. Lab and disease cannot be separated when each lab holds one kind of sample "
       "(NIH-NLM notebooks 46-49b); the merged matrix is for plots only (DISPLAY_ONLY_* files).",
       "- Comparisons are made within a study (notebook 4)."]
open(os.path.join(OUT_DIR, "PROCESSING_LOG_notebook3.md"), "w").write("\n".join(Lg))
print("written to", OUT_DIR, ":")
for f in sorted(os.listdir(OUT_DIR)):
    fp = os.path.join(OUT_DIR, f)
    if os.path.isfile(fp): print(f"  {f:46s} {os.path.getsize(fp)/1e6:8.1f} MB")

---
## ✅ Done. Review order

1. **Part 3:** does our cohort match what notebook 2 printed?
2. **Checkpoint 4 / Part 5:** how many JDM-lab samples have trusted demographics, and which key-sheet rows are unmatched?
3. **Checkpoint 6:** the QN rebuild should give r about 1.0; shared samples should correlate highly.
4. **Checkpoint 7:** the size of the lab effect (display only).
5. **Next:** notebook 4 reads `jdm_lab_samples.csv` and `jdm_lab_expression_log2QN.csv` and builds WGCNA and tests **within** each study.